# Run weaviate in a docker container

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
x = os.environ.get("OPENAI_API_KEY")

In [1]:
!docker ps

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


https://docs.weaviate.io/deploy/installation-guides/docker-installation?utm_source=chatgpt.com

In [7]:
%%writefile docker-compose.yml
services:
  weaviate:
    image: cr.weaviate.io/semitechnologies/weaviate:1.34.5
    command: >
      --host 0.0.0.0
      --port 8080
      --scheme http
    ports:
      - "8080:8080"    # REST API
      - "50051:50051"  # gRPC API
    volumes:
      - weaviate_data:/var/lib/weaviate
    restart: on-failure:0
    environment:
      QUERY_DEFAULTS_LIMIT: 25
      AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED: 'true'
      PERSISTENCE_DATA_PATH: '/var/lib/weaviate'
      CLUSTER_HOSTNAME: 'node1'

volumes:
  weaviate_data:

Overwriting docker-compose.yml


In [8]:
!docker-compose up -d

[+] Running 1/2
 ✔ Network rag-course-deeplearning_default       Created                   0.1s 
 ⠋ Volume rag-course-deeplearning_weaviate_data  Creating                  0.0s 
[+] Running 2/3
 ✔ Network rag-course-deeplearning_default       Created                   0.1s 
 ✔ Volume rag-course-deeplearning_weaviate_data  Created                   0.0s 
 ⠋ Container rag-course-deeplearning-weaviate-1  Creating                  0.1s 
[+] Running 2/3
 ✔ Network rag-course-deeplearning_default       Created                   0.1s 
 ✔ Volume rag-course-deeplearning_weaviate_data  Created                   0.0s 
 ⠙ Container rag-course-deeplearning-weaviate-1  Creating                  0.2s 
[+] Running 2/3
 ✔ Network rag-course-deeplearning_default       Created                   0.1s 
 ✔ Volume rag-course-deeplearning_weaviate_data  Created                   0.0s 
 ⠹ Container rag-course-deeplearning-weaviate-1  Starting                  0.3s 
[+] Running 2/3
 ✔ Network rag-course-deeplea

In [9]:
!docker ps

CONTAINER ID   IMAGE                                             COMMAND                  CREATED         STATUS         PORTS                                                                                          NAMES
699dd7847f34   cr.weaviate.io/semitechnologies/weaviate:1.34.5   "/bin/weaviate --hos…"   6 seconds ago   Up 5 seconds   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp, 0.0.0.0:50051->50051/tcp, [::]:50051->50051/tcp   rag-course-deeplearning-weaviate-1


In [10]:
!curl http://localhost:8080/v1/meta

{"grpcMaxMessageSize":104858000,"hostname":"http://[::]:8080","modules":{"generative-anthropic":{"documentationHref":"https://docs.anthropic.com/en/api/getting-started","name":"Generative Search - Anthropic"},"generative-anyscale":{"documentationHref":"https://docs.anyscale.com/endpoints/overview","name":"Generative Search - Anyscale"},"generative-aws":{"documentationHref":"https://docs.aws.amazon.com/bedrock/latest/APIReference/welcome.html","name":"Generative Search - AWS"},"generative-cohere":{"documentationHref":"https://docs.cohere.com/reference/chat","name":"Generative Search - Cohere"},"generative-contextualai":{"documentationHref":"https://docs.contextual.ai/api-reference/generate/generate","name":"Generative Search - Contextual AI"},"generative-databricks":{"documentationHref":"https://docs.databricks.com/en/machine-learning/foundation-models/api-reference.html#completion-task","name":"Generative Search - Databricks"},"generative-friendliai":{"documentationHref":"https://docs.

In [11]:
!curl -i http://localhost:8080/v1/.well-known/ready

HTTP/1.1 200 OK
Date: Tue, 21 Apr 2026 18:07:43 GMT
Content-Length: 0



## Connect to weaviate

https://academy.weaviate.io/courses/wa101t-py/m3/p3

- using context manager
- or without

In [13]:
import weaviate

with weaviate.connect_to_local(skip_init_checks=True) as client:
    print("Ready:", client.is_ready())
    # do your stuff
# automatically closes connection here


Ready: True


In [14]:
import weaviate
import os

headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

In [68]:
import json

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)

metainfo = client.get_meta()
print(json.dumps(metainfo, indent=2))  # Print the meta information in a readable format

{
  "grpcMaxMessageSize": 104858000,
  "hostname": "http://[::]:8080",
  "modules": {
    "generative-anthropic": {
      "documentationHref": "https://docs.anthropic.com/en/api/getting-started",
      "name": "Generative Search - Anthropic"
    },
    "generative-anyscale": {
      "documentationHref": "https://docs.anyscale.com/endpoints/overview",
      "name": "Generative Search - Anyscale"
    },
    "generative-aws": {
      "documentationHref": "https://docs.aws.amazon.com/bedrock/latest/APIReference/welcome.html",
      "name": "Generative Search - AWS"
    },
    "generative-cohere": {
      "documentationHref": "https://docs.cohere.com/reference/chat",
      "name": "Generative Search - Cohere"
    },
    "generative-contextualai": {
      "documentationHref": "https://docs.contextual.ai/api-reference/generate/generate",
      "name": "Generative Search - Contextual AI"
    },
    "generative-databricks": {
      "documentationHref": "https://docs.databricks.com/en/machine-le

# Create or connect to a collection

In [69]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import os


if not client.collections.exists("Movies"):     
    client.collections.create(
        name="Movies",
        properties=[
            Property(name="title", data_type=DataType.TEXT),
            Property(name="overview", data_type=DataType.TEXT),
            Property(name="vote_average", data_type=DataType.NUMBER),
            Property(name="genre_ids", data_type=DataType.INT_ARRAY),
            Property(name="release_date", data_type=DataType.DATE),
            Property(name="tmdb_id", data_type=DataType.INT),
        ],
        # Define the vectorizer module
        vector_config=Configure.Vectors.text2vec_openai(model="text-embedding-3-small"),
        # Define the generative module
        generative_config=Configure.Generative.openai(model="gpt-4.1-mini")
    )
movies = client.collections.get("Movies")

#client.close()

In [78]:
sample_movies = movies.query.fetch_objects(
    limit=2, 
    include_vector=False) # set to True to return embedding vector

if sample_movies:
    print(f"Thera are {len(movies)} entries in the database")
else:
    print("The database is empty")

sample_movies.objects[0]

Thera are 680 entries in the database


Object(uuid=_WeaviateUUIDInt('006e0bc4-1963-5958-a040-17f3b534c84c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'genre_ids': [18], 'overview': 'In 2003, Harvard undergrad and computer genius Mark Zuckerberg begins work on a new concept that eventually turns into the global social network known as Facebook. Six years later, he is one of the youngest billionaires ever, but Zuckerberg finds that his unprecedented success leads to both personal and legal complications when he ends up on the receiving end of two lawsuits, one involving his former friend.', 'tmdb_id': 37799, 'title': 'The Social Network', 'vote_average': 7.4, 'release_date': datetime.datetime(2010, 10, 1, 0, 0, tzinfo=datetime.timezone.utc)}, references=None, vector={}, collection='Movies')

## Ingest data into weaviate

In [63]:
import weaviate
import pandas as pd
import requests
from datetime import datetime, timezone
import json
from weaviate.util import generate_uuid5
from tqdm.notebook import tqdm

headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

data_url = "https://raw.githubusercontent.com/weaviate-tutorials/edu-datasets/main/movies_data_1990_2024.json"
resp = requests.get(data_url)
df = pd.DataFrame(resp.json())

# Configure collection object
movies = client.collections.use("Movies")

# Enter context manager
with movies.batch.fixed_size(batch_size=200) as batch:
    # Loop through the data
    for i, movie in tqdm(df.iterrows()):
        # Convert data types
        # Convert a JSON date to `datetime` and add time zone information
        release_date = datetime.fromisoformat(movie["release_date"]).replace(tzinfo=timezone.utc)
        # Convert a JSON array to a list of integers
        genre_ids = json.loads(movie["genre_ids"])

        # Build the object payload
        movie_obj = {
            "title": movie["title"],
            "overview": movie["overview"],
            "vote_average": movie["vote_average"],
            "genre_ids": genre_ids,
            "release_date": release_date,
            "tmdb_id": movie["id"],
        }

        # Add object to batch queue
        batch.add_object(
            properties=movie_obj,
            uuid=generate_uuid5(movie["id"])
        )
        # Batcher automatically sends batches

# Check for failed objects
if len(movies.batch.failed_objects) > 0:
    print(f"Failed to import {len(movies.batch.failed_objects)} objects")

client.close()

0it [00:00, ?it/s]

**Note** - weaviate creates ONE embedding for all TEXT fields. It concatenates them an vectorizes them

In [49]:
df.head(1)

,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,/3Nn5BOM1EVw1IYrv6MsbOS6N1Ol.jpg,"[14, 18, 10749]",162,en,Edward Scissorhands,A small suburban town receives a visit from a ...,45.694,/1RFIbuW9Z3eN9Oxw2KaQG5DfLmD.jpg,1990-12-07,Edward Scissorhands,False,7.7,12305


## Semantic search

In [93]:
import weaviate
from weaviate.classes.query import Filter, MetadataQuery
import os

headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

movies = client.collections.use("Movies")

# Perform query
response = movies.query.near_text(
    query="dystopian future",
    limit=3,
    return_metadata=MetadataQuery(distance=True,explain_score=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(o.properties['overview'])
    print(f"Distance to query: {o.metadata.distance:.3f}\n")  # Print the distance of the object from the query
    print(o.metadata)

client.close()

Mad Max: Fury Road 2015
An apocalyptic story set in the furthest reaches of our planet, in a stark desert landscape where humanity is broken, and most everyone is crazed fighting for the necessities of life. Within this world exist two rebels on the run who just might be able to restore order.
Distance to query: 0.546

MetadataReturn(creation_time=None, last_update_time=None, distance=0.5459623336791992, certainty=None, score=None, explain_score='', is_consistent=None, rerank_score=None)
In Time 2011
In the not-too-distant future, the aging gene has been switched off. To avoid overpopulation, time has become the currency and the way people pay for luxuries and necessities. The rich can live forever, while the rest try to negotiate for their immortality. A poor young man who comes into a fortune of time is too late to help his mother from dying. He ends up on the run from a corrupt police force known as the "time keepers".
Distance to query: 0.556

MetadataReturn(creation_time=None, las

## Keyword search

In [87]:
import weaviate
from weaviate.classes.query import Filter, MetadataQuery
import os


# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)
headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.query.bm25(
    query="history", limit=5, return_metadata=MetadataQuery(score=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"BM25 score: {o.metadata.score:.3f}\n")  # Print the BM25 score of the object from the query

client.close()

A Beautiful Mind 2001
BM25 score: 2.723

Legends of the Fall 1994
BM25 score: 2.483

Night at the Museum 2006
BM25 score: 2.412

Hacksaw Ridge 2016
BM25 score: 2.367

The Butterfly Effect 2004
BM25 score: 2.202



## Hybrid search

In [95]:
import weaviate
from weaviate.classes.query import Filter, MetadataQuery
import os

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)
headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.query.hybrid(
    query="history", limit=5, return_metadata=MetadataQuery(score=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"Hybrid score: {o.metadata.score:.3f}\n")  # Print the hybrid search score of the object from the query

client.close()

GoodFellas 1990
Hybrid score: 0.700

A Beautiful Mind 2001
Hybrid score: 0.665

The Butterfly Effect 2004
Hybrid score: 0.599

Hidden Figures 2016
Hybrid score: 0.459

Oppenheimer 2023
Hybrid score: 0.427



## Single Prompt generation

In [61]:
client.close()

In [2]:
!docker-compose down -v

[+] Running 0/1
 ⠋ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.0s 
[+] Running 0/1
 ⠙ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.1s 
[+] Running 0/1
 ⠹ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.2s 
[+] Running 0/1
 ⠸ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.3s 
[+] Running 0/1
 ⠼ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.4s 
[+] Running 0/1
 ⠴ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.5s 
[+] Running 0/1
 ⠦ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.6s 
[+] Running 0/1
 ⠧ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.7s 
[+] Running 0/1
 ⠇ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.8s 
[+] Running 0/1
 ⠏ Container end-to-endragwithweaviate-weaviate-1  Stopping                0.9s 
[+] Running 0/1
 ⠋ Container e